In [1]:
import pandas as pd
import numpy as np

---

### **| Fase 1 ) -** Pré-Processamento

---

In [2]:
df_fair = pd.read_excel('Data/05_Sheets.xlsx')
df_fair.head()

,id,nome,url,repo,is_duplicate,key_value,parent_id
0,CRED-001,Statlog (German Credit Data),https://archive.ics.uci.edu/dataset/144/statlo...,UCI Repository,Yes,144,CRED-002
1,CRED-002,South German Credit,https://www.kaggle.com/datasets/sid321axn/sout...,Kaggle,Ofc,sid321axn/south-german-credit-updated,NaN
2,CRED-003,Default of Credit Card Clients,https://archive.ics.uci.edu/dataset/350/defaul...,UCI Repository,Mod,350,CRED-012
3,CRED-004,Australian Credit Approval,https://www.kaggle.com/datasets/bfueojjsjdjsl/...,Kaggle,No,bfueojjsjdjsl/australian-credit-approval-data-set,NaN
4,CRED-005,Japanese Credit Screening,https://www.kaggle.com/datasets/xiangshan1989/...,Kaggle,No,xiangshan1989/japanese-credit-screening-from-t...,NaN


In [3]:
total_inicial = len(df_fair)
print(f"| 1. Registros totais encontrados via API: {total_inicial}")

df_cleaned = df_fair.dropna(subset=['url']).copy()
total_pos_url = len(df_cleaned)
removidos_url = total_inicial - total_pos_url
print(f"| 2. Removidos por ausência de URL/Documentação: {removidos_url}")
print(f"|    -> Restantes: {total_pos_url}")

mask_originalidade = (df_cleaned['is_duplicate'] == 'No') | (df_cleaned['is_duplicate'] == 'Ofc')
df_cleaned = df_cleaned[mask_originalidade].copy()
total_pos_duplicata = len(df_cleaned)
removidos_duplicata = total_pos_url - total_pos_duplicata
print(f"| 3. Removidos por redundância/duplicação: {removidos_duplicata}")
print(f"|    -> Restantes: {total_pos_duplicata}")

df_cleaned['parent_id'] = df_cleaned['parent_id'].fillna('NaN')
df_cleaned = df_cleaned.reset_index(drop=True)

print(f"| RESULTADO FINAL: {len(df_cleaned)} datasets prontos para análise.")

| 1. Registros totais encontrados via API: 81
| 2. Removidos por ausência de URL/Documentação: 2
|    -> Restantes: 79
| 3. Removidos por redundância/duplicação: 18
|    -> Restantes: 61
| RESULTADO FINAL: 61 datasets prontos para análise.


---

##### | 1.1 ) - Kaggle

In [4]:
df_cleaned_kaggle = df_cleaned[df_cleaned['repo'] == 'Kaggle']
df_cleaned_kaggle.head()

,id,nome,url,repo,is_duplicate,key_value,parent_id
0,CRED-002,South German Credit,https://www.kaggle.com/datasets/sid321axn/sout...,Kaggle,Ofc,sid321axn/south-german-credit-updated,NaN
1,CRED-004,Australian Credit Approval,https://www.kaggle.com/datasets/bfueojjsjdjsl/...,Kaggle,No,bfueojjsjdjsl/australian-credit-approval-data-set,NaN
2,CRED-005,Japanese Credit Screening,https://www.kaggle.com/datasets/xiangshan1989/...,Kaggle,No,xiangshan1989/japanese-credit-screening-from-t...,NaN
3,CRED-007,Polish Companies Bankruptcy,https://www.kaggle.com/datasets/stealthtechnol...,Kaggle,No,stealthtechnologies/predict-bankruptcy-in-poland,NaN
9,CRED-022,Home Credit Default Risk,https://www.kaggle.com/c/home-credit-default-risk,Kaggle,No,competitions/home-credit-default-risk/data,NaN


---

##### | 1.2 ) - UCI Repository	

In [5]:
df_cleaned_uci = df_cleaned[df_cleaned['repo'] == 'UCI Repository']
df_cleaned_uci.head()

,id,nome,url,repo,is_duplicate,key_value,parent_id


---

##### | 1.3 ) - Open ML

In [6]:
df_cleaned_openml = df_cleaned[df_cleaned['repo'] == 'OpenML']
df_cleaned_openml.head()

,id,nome,url,repo,is_duplicate,key_value,parent_id
4,CRED-008,Qualitative Bankruptcy,https://www.kaggle.com/datasets/jagadeesh23/qu...,OpenML,No,1495,NaN
5,CRED-010,credit-approval,https://www.openml.org/d/29,OpenML,Ofc,29,NaN
6,CRED-012,default-of-credit-card-clients v1,https://www.openml.org/d/42477,OpenML,Ofc,42477,NaN
7,CRED-020,LoanDefaultPrediction,https://www.openml.org/d/44067,OpenML,No,44067,NaN
8,CRED-021,bank-marketing,https://www.openml.org/d/44074,OpenML,No,44074,NaN


---

### **| Fase 2 ) -** Extração

---

In [7]:
import pandas as pd
import numpy as np
import os
import shutil
import zipfile
import locale
import openml
from ucimlrepo import fetch_ucirepo
from kaggle.api.kaggle_api_extended import KaggleApi

In [8]:
try:
    locale.setlocale(locale.LC_TIME, 'C')
except locale.Error:
    pass 

# =====================================================================
# | FUNÇÕES AUXILIARES E DE TRATAMENTO
# =====================================================================

def is_encrypted(mean_length, threshold=5):
    """Verifica ofuscação pelo tamanho médio do nome das colunas."""
    return mean_length < threshold if pd.notnull(mean_length) else 'N/A'

def get_columns_size_mean(columns):
    """Calcula o tamanho médio dos nomes das colunas."""
    if len(columns) == 0: return 0
    return np.mean([len(str(c)) for c in columns])


def consolidate_relational_data(path, files):
    """
    Consolida múltiplos CSVs através de análise estrutural inteligente.
    Distingue divisões de Machine Learning (Train/Test) de Tabelas Relacionais.
    """
    if len(files) == 1:
        return pd.read_csv(os.path.join(path, files[0]), low_memory=False), files[0]
    
    dfs = {}
    for f in files:
        try:
            dfs[f] = pd.read_csv(os.path.join(path, f), low_memory=False)
        except Exception:
            continue
            
    if not dfs:
        return pd.DataFrame(), None

    # Identifica a tabela mestre pelo maior número de instâncias (linhas)
    main_file = max(dfs, key=lambda k: dfs[k].shape[0])
    df_master = dfs.pop(main_file)
    merged_files = [main_file]
    
    for file_name, df_aux in dfs.items():
        colunas_master = set(df_master.columns)
        colunas_aux = set(df_aux.columns)
        common_cols = list(colunas_master & colunas_aux)
        
        if not common_cols: continue

        # 1. Análise de Sobreposição (Schema Overlap)
        # Verifica qual a porcentagem de colunas em comum em relação à menor tabela
        overlap_ratio = len(common_cols) / min(len(colunas_master), len(colunas_aux))
        
        # 2. DECISÃO LÓGICA: Se são altamente similares (>= 70%), são splits (Treino/Teste)
        if overlap_ratio >= 0.70:
            # Empilha os dados verticalmente. 
            # Aumenta as linhas, não polui as colunas e descarta colunas exclusivas irrelevantes.
            df_master = pd.concat([df_master, df_aux], join='inner', ignore_index=True)
            merged_files.append(file_name)
        
        # 3. DECISÃO LÓGICA: Se têm baixa similaridade, mas têm chaves (ID), são relacionais
        else:
            join_keys = [c for c in common_cols if 'id' in c.lower() or 'key' in c.lower()]

            if join_keys:
                # Trata relação 1-para-Muitos antes de fundir
                if df_aux.shape[0] > df_master.shape[0] * 1.1:
                    agg_logic = {}
                    for col in df_aux.columns:
                        if col in join_keys: continue
                        if pd.api.types.is_numeric_dtype(df_aux[col]):
                            agg_logic[col] = 'mean'
                        else:
                            agg_logic[col] = 'first'
                    
                    df_aux = df_aux.groupby(join_keys[0]).agg(agg_logic).reset_index()
                
                # Funde horizontalmente (Left Join)
                df_master = pd.merge(df_master, df_aux, on=join_keys, how='left', suffixes=('', f'_{file_name}'))
                merged_files.append(file_name)
            
    return df_master, ", ".join(merged_files)


In [9]:
# Lista multilingue de atributos sensíveis
SENSITIVE_TERMS = ['age', 'gender', 'sex', 'race', 'ethnicity', 'status', 'marital',]

In [10]:
# =====================================================================
# | FASE 2 ) - EXTRAÇÃO AUTOMÁTICA E PROCESSAMENTO (BLINDADO)
# =====================================================================

# --- 2.1 Processamento Kaggle ---
def process_kaggle_datasets(df_input, path='./DataBuffer'):
    api = KaggleApi()
    api.authenticate()
    results_list = []

    for index, row in df_input.iterrows():
        dataset_id = row['id']
        ref = str(row['key_value'])
        os.makedirs(path, exist_ok=True)
        
        # Estrutura base de resposta (usada em caso de erro ou sucesso)
        row_data = {
            'id': dataset_id, 'User_Ref': ref, 'Dataset_Title': row['nome'], 'URL': row['url'],
            'is_duplicate': row['is_duplicate'], 'parent_id': str(row['parent_id']).replace('[', '').replace(']', ''),
            'Source': 'Kaggle'
        }

        try:
            # TRAVA DE SEGURANÇA: Bloqueio direto dos datasets que causam "Explosão Cartesiana" na RAM
            if dataset_id in ['CRED-049', 'CRED-059', 'CRED-057', 'CRED-064']:
                raise Exception("Dataset massivo/complexo. Processamento bloqueado para proteger a memória RAM.")

            if ref.startswith('competitions/'):
                comp_name = ref.replace('competitions/', '')
                api.competition_download_files(comp_name, path=path)
                for item in os.listdir(path):
                    if item.endswith('.zip'):
                        with zipfile.ZipFile(os.path.join(path, item), 'r') as zip_ref:
                            zip_ref.extractall(path)
                        os.remove(os.path.join(path, item))
            else:
                # O download direto ocorre, mas os problemáticos já foram barrados acima
                api.dataset_download_files(ref, path=path, unzip=True)
                
            files = [f for root, _, f_names in os.walk(path) for f in f_names if f.endswith('.csv')]
            if not files: raise Exception("Nenhum arquivo CSV encontrado após extração.")
            
            df_temp, file_info = consolidate_relational_data(path, files)
            if df_temp.empty: raise Exception("Falha na consolidação dos dados relacionais.")

            columns = df_temp.columns.values
            found_sensitive = [col for col in columns if any(s in str(col).lower() for s in SENSITIVE_TERMS)]
            
            row_data.update({
                'Status_Processamento': 'Sucesso', 'Observacao': 'OK', 'File_Name': file_info,
                'is_encrypted': is_encrypted(mean_length=get_columns_size_mean(columns), threshold=5),
                'Columns': list(columns), 'Columns_Count': df_temp.shape[1], 'Rows_Count': df_temp.shape[0],
                'Rows_With_Missing': df_temp.isnull().any(axis=1).sum(), 'Missing_Values': df_temp.isnull().sum().sum(),
                'Numeric_Cols': len(df_temp.select_dtypes(include=['number']).columns),
                'Categorical_Cols': len(df_temp.select_dtypes(exclude=['number']).columns),
                'Sensitive_Columns': found_sensitive,
                'Memory_Usage_MB': round(df_temp.memory_usage(deep=True).sum() / (1024**2), 2)
            })
            print(f"| {dataset_id} processado com sucesso.")
            del df_temp # Libera a RAM imediatamente após a extração das métricas

        except Exception as e:
            print(f"| {dataset_id} -> Erro Técnico: {e}")
            row_data.update({
                'Status_Processamento': 'Erro Técnico', 'Observacao': str(e), 'File_Name': 'N/A',
                'is_encrypted': 'N/A', 'Columns': 'N/A', 'Columns_Count': 0, 'Rows_Count': 0,
                'Rows_With_Missing': 0, 'Missing_Values': 0, 'Numeric_Cols': 0, 'Categorical_Cols': 0,
                'Sensitive_Columns': [], 'Memory_Usage_MB': 0
            })
        
        results_list.append(row_data)
        shutil.rmtree(path, ignore_errors=True)

    return pd.DataFrame(results_list)

# --- 2.2 Processamento OpenML ---
def process_openml_datasets(df_input):
    results_list = []
    for index, row in df_input.iterrows():
        dataset_id = row['id']
        row_data = {
            'id': dataset_id, 'User_Ref': row['key_value'], 'Dataset_Title': row['nome'], 'URL': row['url'],
            'is_duplicate': row['is_duplicate'], 'parent_id': str(row['parent_id']).replace('[', '').replace(']', ''),
            'Source': 'OpenML', 'File_Name': 'API_Object'
        }

        try:
            dataset = openml.datasets.get_dataset(row['key_value'])
            df_temp, _, _, _ = dataset.get_data(dataset_format="dataframe")
            
            columns = df_temp.columns.values
            found_sensitive = [col for col in columns if any(s in str(col).lower() for s in SENSITIVE_TERMS)]
            
            row_data.update({
                'Status_Processamento': 'Sucesso', 'Observacao': 'OK',
                'is_encrypted': is_encrypted(mean_length=get_columns_size_mean(columns), threshold=5),
                'Columns': list(columns), 'Columns_Count': df_temp.shape[1], 'Rows_Count': df_temp.shape[0],
                'Rows_With_Missing': df_temp.isnull().any(axis=1).sum(), 'Missing_Values': df_temp.isnull().sum().sum(),
                'Numeric_Cols': len(df_temp.select_dtypes(include=['number']).columns),
                'Categorical_Cols': len(df_temp.select_dtypes(exclude=['number']).columns),
                'Sensitive_Columns': found_sensitive,
                'Memory_Usage_MB': round(df_temp.memory_usage(deep=True).sum() / (1024**2), 2)
            })
            print(f"| {dataset_id} processado com sucesso.")
            del df_temp # Libera a RAM
            
        except Exception as e:
            print(f"| {dataset_id} -> Erro Técnico: {e}")
            row_data.update({
                'Status_Processamento': 'Erro Técnico', 'Observacao': str(e),
                'is_encrypted': 'N/A', 'Columns': 'N/A', 'Columns_Count': 0, 'Rows_Count': 0,
                'Rows_With_Missing': 0, 'Missing_Values': 0, 'Numeric_Cols': 0, 'Categorical_Cols': 0,
                'Sensitive_Columns': [], 'Memory_Usage_MB': 0
            })
        
        results_list.append(row_data)
    return pd.DataFrame(results_list)

# --- 2.3 Processamento UCI ---
def process_uci_datasets(df_input):
    results_list = []
    for index, row in df_input.iterrows():
        dataset_id = row['id']
        row_data = {
            'id': dataset_id, 'User_Ref': row['key_value'], 'Dataset_Title': row['nome'], 'URL': row['url'],
            'is_duplicate': row['is_duplicate'], 'parent_id': str(row['parent_id']).replace('[', '').replace(']', ''),
            'Source': 'UCI', 'File_Name': 'API_Object'
        }

        try:
            dataset = fetch_ucirepo(id=int(row['key_value']))
            df_temp = dataset.data.features
            if df_temp is None: raise Exception("Funcionalidades (features) não encontradas no dataset UCI.")
            
            columns = df_temp.columns.values
            found_sensitive = [col for col in columns if any(s in str(col).lower() for s in SENSITIVE_TERMS)]
            
            row_data.update({
                'Status_Processamento': 'Sucesso', 'Observacao': 'OK',
                'is_encrypted': is_encrypted(mean_length=get_columns_size_mean(columns), threshold=5),
                'Columns': list(columns), 'Columns_Count': df_temp.shape[1], 'Rows_Count': df_temp.shape[0],
                'Rows_With_Missing': df_temp.isnull().any(axis=1).sum(), 'Missing_Values': df_temp.isnull().sum().sum(),
                'Numeric_Cols': len(df_temp.select_dtypes(include=['number']).columns),
                'Categorical_Cols': len(df_temp.select_dtypes(exclude=['number']).columns),
                'Sensitive_Columns': found_sensitive,
                'Memory_Usage_MB': round(df_temp.memory_usage(deep=True).sum() / (1024**2), 2)
            })
            print(f"| {dataset_id} processado com sucesso.")
            del df_temp # Libera a RAM
            
        except Exception as e:
            print(f"| {dataset_id} -> Erro Técnico: {e}")
            row_data.update({
                'Status_Processamento': 'Erro Técnico', 'Observacao': str(e),
                'is_encrypted': 'N/A', 'Columns': 'N/A', 'Columns_Count': 0, 'Rows_Count': 0,
                'Rows_With_Missing': 0, 'Missing_Values': 0, 'Numeric_Cols': 0, 'Categorical_Cols': 0,
                'Sensitive_Columns': [], 'Memory_Usage_MB': 0
            })
        
        results_list.append(row_data)
    return pd.DataFrame(results_list)

---

### **| Fase 3 ) -** Execução

---

In [11]:
print("--- Iniciando FASE 2: Extração Kaggle ---")
df_info_kaggle = process_kaggle_datasets(df_input=df_cleaned_kaggle)

print("\n--- Iniciando FASE 2: Extração OpenML ---")
df_info_openml = process_openml_datasets(df_input=df_cleaned_openml)

print("\n--- Iniciando FASE 2: Extração UCI ---")
df_info_uci = process_uci_datasets(df_input=df_cleaned_uci)

--- Iniciando FASE 2: Extração Kaggle ---
Dataset URL: https://www.kaggle.com/datasets/sid321axn/south-german-credit-updated
| CRED-002 processado com sucesso.
Dataset URL: https://www.kaggle.com/datasets/bfueojjsjdjsl/australian-credit-approval-data-set
| CRED-004 processado com sucesso.
Dataset URL: https://www.kaggle.com/datasets/xiangshan1989/japanese-credit-screening-from-the-uc-irvine
| CRED-005 processado com sucesso.
Dataset URL: https://www.kaggle.com/datasets/stealthtechnologies/predict-bankruptcy-in-poland
| CRED-007 processado com sucesso.
| CRED-022 -> Erro Técnico: 403 Client Error: Forbidden for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles
Dataset URL: https://www.kaggle.com/datasets/wordsforthewise/lending-club
| CRED-023 -> Erro Técnico: Falha na consolidação dos dados relacionais.
| CRED-025 -> Erro Técnico: 403 Client Error: Forbidden for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles
Dataset

In [12]:
# 3.1 Unificação dos DataFrames
df_final = pd.concat([df_info_kaggle, df_info_openml, df_info_uci], ignore_index=True)

In [13]:
df_final['Sparsity_Ratio_%'] = np.where(
    df_final['Rows_Count'] > 0,
    np.round((df_final['Missing_Values'] / (df_final['Rows_Count'] * df_final['Columns_Count'])) * 100, 2),
    0
)

df_final['Dimensionality_Ratio'] = np.where(
    df_final['Rows_Count'] > 0,
    np.round(df_final['Columns_Count'] / df_final['Rows_Count'], 6),
    0
)

df_final['Has_Sensitive_Data'] = df_final['Sensitive_Columns'].apply(
    lambda x: len(x) > 0 if isinstance(x, list) else False
)

df_final['Sensitive_Count'] = df_final['Sensitive_Columns'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

In [14]:
df_final = df_final.sort_values(by='id').reset_index(drop=True)
os.makedirs('Data', exist_ok=True)
caminho_saida = 'Data/06_Datasets_Metadata_Final.xlsx'
df_final.to_excel(caminho_saida, index=False, engine='openpyxl')

print(f"| Total de Registros: {len(df_final)}")
print(f"| Sucessos: {len(df_final[df_final['Status_Processamento'] == 'Sucesso'])}")
print(f"| Erros/Bloqueios: {len(df_final[df_final['Status_Processamento'] == 'Erro Técnico'])}")
print(f"| Arquivo salvo em: {caminho_saida}")

| Total de Registros: 61
| Sucessos: 53
| Erros/Bloqueios: 8
| Arquivo salvo em: Data/06_Datasets_Metadata_Final.xlsx


In [15]:
import pandas as pd
import os

caminho_entrada = 'Data/06_Datasets_Metadata_Final.xlsx'
df_completo = pd.read_excel(caminho_entrada, engine='openpyxl')

df_erros = df_completo[df_completo['Status_Processamento'] == 'Erro Técnico'].copy()
df_original = df_completo[df_completo['Status_Processamento'] != 'Erro Técnico'].copy()

colunas_remover = ['File_Name', 'Status_Processamento', 'Source', 'parent_id', 'is_duplicate', 'Observacao', 'User_Ref']
df_original = df_original.drop(columns=colunas_remover, errors='ignore')

caminho_saida_original = 'Data/06_Datasets_Metadata_Limpo.xlsx'
caminho_saida_erros = 'Data/07_Datasets_Errors_Final.xlsx'

df_original.to_excel(caminho_saida_original, index=False, engine='openpyxl')
df_erros.to_excel(caminho_saida_erros, index=False, engine='openpyxl')

print(f"| Registros originais com sucesso ({len(df_original)}): Salvos em {caminho_saida_original}")
print(f"| Registros de erro isolados ({len(df_erros)}): Salvos em {caminho_saida_erros}")

| Registros originais com sucesso (53): Salvos em Data/06_Datasets_Metadata_Limpo.xlsx
| Registros de erro isolados (8): Salvos em Data/07_Datasets_Errors_Final.xlsx
